# UPS Strategy Optimization Visual Guide

This notebook walks through:
1. Baseline backtest run (one fixed config)
2. `bt.optimize(...)` grid search for best parameters
3. Viewing best vs full combination table

In [1]:
from fetch_data.fetch_bybit_candles import load_ohlcv_bybit

df= load_ohlcv_bybit()


df

,Open,High,Low,Close,Volume
Date,,,,,
2020-03-25 00:00:00+00:00,6500.0,6745.5,6500.0,6698.5,1809.520
2020-03-26 00:00:00+00:00,6698.5,6767.0,6512.0,6733.5,3904.964
2020-03-27 00:00:00+00:00,6733.5,6838.0,6235.0,6354.0,4605.347
2020-03-28 00:00:00+00:00,6354.0,6354.0,6010.0,6230.5,5750.959
2020-03-29 00:00:00+00:00,6230.5,6247.0,5858.0,5873.0,5347.238
...,...,...,...,...,...
2020-10-06 00:00:00+00:00,10779.0,10782.0,10531.0,10600.0,5979.297
2020-10-07 00:00:00+00:00,10600.0,10668.0,10554.0,10656.5,2586.677
2020-10-08 00:00:00+00:00,10656.5,10945.0,10532.0,10906.0,6515.567


In [ ]:
from ups_backtest import Settings, UPSStrategy, load_ohlcv_kucoin, run 
from backtesting.lib import FractionalBacktest

# 1) Load data once
df = load_ohlcv_kucoin(
    symbol="XBTUSDTM",
    market_type="futures",
    timeframe="15min",
    start_time="2026-01-01 00:00:00",
    end_time=None,
)

print("[Step 1] Data loaded. Rows:", len(df))

SyntaxError: trailing comma not allowed without surrounding parentheses (1325195101.py, line 1)

In [ ]:
# 2) Baseline run
baseline = Settings(
    ma_length=50,
    stop_multiplier=1.0,
    risk_reward_multiplier=1.0,
    trail_stop=True,
    trail_stop_size=1.0,
)

_, baseline_stats = run(df, settings=baseline)
print("[Step 2] Baseline results:\n", baseline_stats[["Return [%]", "# Trades", "Win Rate [%]", "SQN"]])

[Step 2] Baseline results:
 Return [%]     -10.442487
# Trades               84
Win Rate [%]    26.190476
SQN             -1.045682
dtype: object


In [ ]:
# 3) Optimization run
bt = FractionalBacktest(
    df,
    UPSStrategy,
    cash=10_000,
    commission=0.001,
    exclusive_orders=True,
    finalize_trades=True,
)

opt, heatmap = bt.optimize(
    ma_length=[20, 30, 40, 50],
    stop_multiplier=[0.8, 1.0, 1.2],
    risk_reward_multiplier=[1.0, 1.5, 2.0],
    trail_stop=[False, True],
    maximize="SQN",
    method="grid",
    return_heatmap=True,
)

print("[Step 3] Best optimize result:\n", opt[["Return [%]", "# Trades", "Win Rate [%]", "SQN"]])
print("Best params:\n", opt["_strategy"])

print("\n[Step 3] Heatmap head (all tested combos):")
print(heatmap.reset_index().head())

/Users/andre/Documents/Python_local/pine_script/.venv/lib/python3.13/site-packages/backtesting/backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()
Backtest.optimize:   0%|          | 0/72 [00:00<?, ?it/s]



























































                                                                    

[Step 3] Best optimize result:
 Return [%]       5.616838
# Trades               67
Win Rate [%]    34.328358
SQN              0.431672
dtype: object
Best params:
 UPSStrategy(ma_length=20,stop_multiplier=1.2,risk_reward_multiplier=2.0,trail_stop=)

[Step 3] Heatmap head (all tested combos):
   ma_length  stop_multiplier  risk_reward_multiplier  trail_stop       SQN
0         20              0.8                     1.0       False -2.711191
1         20              0.8                     1.0        True -0.686065
2         20              0.8                     1.5       False -1.771459
3         20              0.8                     1.5        True -0.563401
4         20              0.8                     2.0       False -1.014604


In [ ]:
# 4) Full table of combos with score
results_df = heatmap.reset_index().rename(columns={0: "SQN"})
results_df = results_df.sort_values(by="SQN", ascending=False)
print("[Step 4] Best 10 combos:")
print(results_df.head(10))

# If you want the trade list of the best run:
print("\n[Step 4] Best run trade table:")
print(opt["_trades"].head(10))

[Step 4] Best 10 combos:
    ma_length  stop_multiplier  risk_reward_multiplier  trail_stop       SQN
57         50              0.8                     1.5        True  1.124113
55         50              0.8                     1.0        True  1.071803
61         50              1.0                     1.0        True  1.045184
42         40              1.0                     1.0       False  0.989986
60         50              1.0                     1.0       False  0.961122
56         50              0.8                     1.5       False  0.954271
66         50              1.2                     1.0       False  0.952396
67         50              1.2                     1.0        True  0.949190
59         50              0.8                     2.0        True  0.863039
58         50              0.8                     2.0       False  0.823543

[Step 4] Best run trade table:
       Size  EntryBar  ExitBar  EntryPrice     ExitPrice            SL   TP  \
0  0.084594      

In [ ]:
opt

Start                     2026-01-01 00:00...
End                       2026-03-23 11:00...
Duration                     81 days 11:00:00
Exposure Time [%]                    30.63547
Equity Final [$]                  10561.68377
Equity Peak [$]                   11007.49821
Commissions [$]                    1269.78977
Return [%]                            5.61684
Buy & Hold Return [%]               -18.49182
Return (Ann.) [%]                    22.80982
Volatility (Ann.) [%]                38.64071
CAGR [%]                             27.74502
Sharpe Ratio                          0.59031
Sortino Ratio                         1.41459
Calmar Ratio                          2.55018
Alpha [%]                             3.69714
Beta                                 -0.10381
Max. Drawdown [%]                    -8.94438
Avg. Drawdown [%]                    -1.72986
Max. Drawdown Duration       34 days 15:30:00
Avg. Drawdown Duration        4 days 06:00:00
# Trades                          